# 20a · Out-of-sample factors for the train period (E20, part 1 of 2)

The E15 GRU was trained on all of train, so its train-period forecasts are **in-sample**: the RL agents learned on
forecasts better than any they will see later, and train could not be used as a walk-forward test period.

This notebook **cross-fits** the factors. Train is cut into **4 consecutive blocks** (about 3.2 months each). For
each block, the E15 configuration (same hyperparameters, same 3 seeds, early stopping on validation as in E15) is
retrained on train **without that block and without 1 day (1440 bars) on each side of it**, so no training label
looks into the block (labels look up to 900 bars ahead). The block's factors come only from those 3 models,
averaged as in E15. Validation, test and stress_bear keep the E15 factors.

Caveat: this is a purged K-fold, not a pure walk-forward. The model for an early block has also seen **later**
train blocks.

T4 GPU, about 30-40 minutes (12 runs of about 2-3 minutes; one E15 run took 2.3 minutes). Runs resume after a
disconnect. Writes `thesis_rl/initiation/e20/crossfit_mean3/` (the factors notebook 20b uses) and
`report/e20a_crossfit.md`. Test and stress_bear are loaded only because the model reads one continuous series;
their metrics are neither printed nor used.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "20a_crossfit_factors")

In [ ]:
import json, zipfile, numpy as np, pandas as pd
from pathlib import Path
from rltrade import long_context, factor_rule as FR
from rltrade.rnn import crossfit as CF
from rltrade.rnn.train import label_metrics, available_splits

INIT = L.root / "initiation"
E15, OUT, REP = INIT / "e15", INIT / "e20", INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
if not (LOCAL / "fixed_plus" / "meta.json").exists():
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    print("unzipping", ZIP, "->", LOCAL)
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL)
if not (LOCAL / "fixed_plus_long" / "meta.json").exists():
    long_context.build(LOCAL / "fixed_plus", LOCAL / "fixed_plus_long")

def on_drive(p):                                    # paths written on another machine: same place under initiation/
    p = Path(p)
    return p if p.exists() else INIT.joinpath(*p.parts[p.parts.index("initiation") + 1:])
sel = json.loads((INIT / "selected_factors_e15.json").read_text())
BASE = on_drive(sel["factor_dir"])                  # E15 selected_mean3
PARAMS = json.loads((E15 / "best_params.json").read_text())
INPUTS = PARAMS["inputs"]
PREP = LOCAL / INPUTS
SPLITS = available_splits(PREP)
N_TRAIN = len(np.load(PREP / "alpha101_train.npy", mmap_mode="r"))
K, SEEDS, EMBARGO, MAX_MINUTES = 4, (0, 1, 2), CF.EMBARGO, 60
CF_DIR = OUT / "crossfit_mean3"
print(f"E15 config: {PARAMS} | inputs {INPUTS} | train {N_TRAIN:,} rows -> blocks {CF.blocks(N_TRAIN, K)}")

## 1 · Retrain without each block (4 blocks x 3 seeds)

In [ ]:
FOLDS = CF.crossfit_runs(PARAMS, PREP, OUT, N_TRAIN, k=K, seeds=SEEDS, embargo=EMBARGO, max_minutes=MAX_MINUTES)
runs = []
for f, dirs in FOLDS.items():
    for s, d in zip(SEEDS, dirs):
        m = json.loads((d / "metrics.json").read_text())
        runs.append({"held-out block": f, "seed": s, "steps": m["steps_run"], "stop": m["stop_reason"], "minutes": m["minutes"],
                     "val mse": round(m["val"]["mse"], 5), "val ic": round(m["val"]["ic_mean"], 4)})
runs = pd.DataFrame(runs); print(runs.to_string(index=False))
FD = CF.write_factors(FOLDS, N_TRAIN, BASE, CF_DIR, SPLITS, k=K, embargo=EMBARGO)
print("wrote", FD)

## 2 · How in-sample were the old train factors?

In [ ]:
Y = {s: np.load(PREP / f"labels_{s}.npy") for s in ("train", "val")}
OLD, NEW = np.load(BASE / "factors_train.npy"), np.load(FD / "factors_train.npy")
assert OLD.shape == NEW.shape == (N_TRAIN, 8)
price = np.load(LOCAL / "fixed_plus" / "market_train.npz")["midpoint"].astype(np.float64)
rows = []
for f, (lo, hi) in enumerate(CF.blocks(N_TRAIN, K)):
    hi_l = min(hi, len(Y["train"]))
    for tag, F_ in (("E15, in-sample", OLD), ("cross-fitted", NEW)):
        m = label_metrics(F_[lo:hi_l], Y["train"][lo:hi_l])
        r = {"block": f, "factors": tag, "mse": round(m["mse"], 5), "ic mean": round(m["ic_mean"], 4),
             "dir acc": round(m["dir_acc_mean"], 4), **{f"ic f{j}": round(m["ic"][j], 3) for j in range(8)}}
        for j in (4, 5, 6, 7):                        # gross half spread of the strongest 1 % of signals (cf. E19)
            fwd = FR.forward_returns(price[lo:hi], FR.OFFSETS[j])
            cuts = np.quantile(F_[lo:hi, j], (0.01, 0.99))
            mean, _ = FR.edge(F_[lo:hi, j], fwd, cuts)
            r[f"f{j} 1% half spread %"] = round((mean[-1] - mean[0]) / 2, 3)
        rows.append(r)
m = label_metrics(np.load(BASE / "factors_val.npy"), Y["val"])
rows.append({"block": "val", "factors": "E15 (unchanged)", "mse": round(m["mse"], 5), "ic mean": round(m["ic_mean"], 4),
             "dir acc": round(m["dir_acc_mean"], 4), **{f"ic f{j}": round(m["ic"][j], 3) for j in range(8)}})
cmp = pd.DataFrame(rows)
print("label fit by train block (higher IC = forecasts closer to what happened); round-trip cost 0.24 %")
print(cmp.to_string(index=False))

In [ ]:
(REP / "e20a_crossfit.md").write_text(
    "# E20a: cross-fitted train factors\n\n"
    f"Train cut into {K} blocks {CF.blocks(N_TRAIN, K)}; block k predicted by the E15 configuration retrained without "
    f"block k +- {EMBARGO} bars (seeds {SEEDS}, mean of seeds). Other splits = E15 ({BASE.name}).\n\n"
    "## Runs\n\n" + runs.to_markdown(index=False) + "\n\n## In-sample vs cross-fitted, by train block\n\n"
    + cmp.to_markdown(index=False) + f"\n\nFactors: {FD}\n", encoding="utf-8")
print("wrote", REP / "e20a_crossfit.md")

In [ ]:
AUTO_DISCONNECT = True               # release the GPU once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()